In [ ]:
import threading
import numpy as np
import time
from PIL import Image
from constants import PGM
import multiprocessing

[[ 92  55]
 [155 215]]


In [ ]:
def image_negative_worker(new_image, image, start, end, result_lock):
    """Generate a negative for the image chunk of the image"""
    image_chunk = image[start:end].copy()
    new_image_chunk = 255 - image_chunk
    # Need lock when updating shared result
    with result_lock:
        new_image[start:end] += new_image_chunk

In [ ]:
def calculate_image_threads(height, min_rows_per_thread=50, max_threads=16):
    """
    Calculate threads for image processing

    Args:
        height: Image height in pixels
        min_rows_per_thread: Minimum rows to avoid overhead
        max_threads: Upper limit
    """
    cpu_cores = multiprocessing.cpu_count()

    # Maximum threads possible based on rows
    max_threads_by_rows = max(1, height // min_rows_per_thread)

    # Optimal threads (can't have more threads than rows)
    optimal = min(cpu_cores, max_threads_by_rows, max_threads)

    # Keep at least one CPU core free when all cores would be used
    if optimal == cpu_cores and optimal > 1:
        optimal -= 1

    # Ensure each thread gets reasonable work
    if optimal > 1:
        rows_per_thread = height // optimal
        if rows_per_thread < min_rows_per_thread:
            optimal = max(1, height // min_rows_per_thread)
            if optimal == cpu_cores and optimal > 1:
                optimal -= 1

    return optimal


if __name__ == "__main__":
    test_images = [
        (100, 100),      # Small
        (480, 640),      # VGA
        (1080, 1920),    # Full HD
        (2160, 3840),    # 4K
    ]
    
    for height, width in test_images:
        threads = calculate_image_threads(height)
        rows_per_thread = height // threads
        print(f"{height}x{width}: {threads} threads, ~{rows_per_thread} rows/thread")


In [ ]:
def get_image_thread_results(image_path="lena.pgm"):
    with Image.open(image_path) as img:
        arr = np.array(img)
        print(arr)
        data = arr.tobytes()
        metadata = PGM(img.height, img.width, int(arr.max()), data)
    threads = calculate_image_threads(metadata.height)
    return {
        "metadata": metadata,
        "threads": threads,
    }


def get_image_slice_values(image_path="lena.pgm"):
    results = get_image_thread_results(image_path)
    metadata = results["metadata"]

    width, height = metadata.width, metadata.height
    image = np.frombuffer(metadata.data, dtype=np.uint8).reshape(height, width)

    num_threads = results["threads"]
    rows_per_thread = height // num_threads

    slices = []
    for i in range(num_threads):
        start = i * rows_per_thread
        end = (i + 1) * rows_per_thread if i < num_threads - 1 else height
        slices.append([start, end]) # slices

    return {
        "metadata": metadata,
        "image": image,
        "threads": num_threads,
        "slices": slices,
    }


if __name__ == "__main__":
    image_values = get_image_slice_values("lena.pgm")
    metadata = image_values["metadata"]
    image = image_values["image"]
    num_threads = image_values["threads"]
    slices = image_values["slices"]

In [ ]:
if __name__ == "__main__":
    # ---------- SUM EXAMPLE ----------
    print("\n--- SUM WITH 2 THREADS ---")

    # Using list for mutable shared result (or array with single element)
    sum_result = [0.0]  # Shared mutable result
    sum_lock = threading.Lock()

    # Split vector into 2 sections
    mid_point = vector_size // 2

    # Create 2 sum threads
    thread_sum1 = threading.Thread(
        target=sum_section, args=(vector, sum_result, 0, mid_point, sum_lock, 1)
    )
    thread_sum2 = threading.Thread(
        target=sum_section, args=(vector, sum_result, mid_point, vector_size, sum_lock, 2)
    )

    # Start threads
    start_time = time.time()
    thread_sum1.start()
    thread_sum2.start()

    # Wait for completion
    thread_sum1.join()
    thread_sum2.join()

    print(f"Total sum: {sum_result[0]}")
    print(f"Verification (np.sum): {np.sum(vector)}")
    print(f"Time taken: {time.time() - start_time:.4f} seconds")
